# Capítulo 3 — Del portátil a la nube

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** El equipo web de TiendaNova pregunta: *«¿A qué URL llamamos?»*. El agente vive en tu portátil.

| | |
|---|---|
| **🧠 Componente agéntico** | Entorno de ejecución: el agente como servicio |
| **La necesidad** | Que el agente esté disponible para miles de clientes, aislado entre ellos y sin gestionar servidores |
| **Lo que construimos** | Proyecto con la AgentCore CLI y despliegue en **Runtime** |
| **🏠 En local** | ☁️ No aplica: este componente existe porque los clientes están en Internet. |
| **☁️ Por qué AWS al salir a Internet** | Un proceso local es un único punto de fallo: todos los usuarios comparten proceso (y lo que haya en memoria), no escala y muere si cierras el portátil. Runtime ejecuta cada sesión en su propia **microVM**, escala de cero a miles y cobra por uso. |
| **En el ordenador sería…** | **Encender el ordenador para todos**: una máquina disponible 24/7, con cada usuario en su propia sesión aislada |
| **Requisitos del caso** | Base de todos los demás |
| **Prerrequisito · duración** | Capítulo 2 · 60 min |
| **⚠️ Dónde se rompe la analogía** | Más que «un ordenador para todos», Runtime da **un ordenador nuevo a cada sesión**: el aislamiento es la mitad de su valor. |

In [ ]:
# Configuración común: ejecuta siempre esta celda primero
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()

> 🎮 **¿Llegas tarde o algo se rompió?** La celda siguiente comprueba que tienes todo lo necesario. Si no, descomenta `ponerse_al_dia(3)`: carga la partida guardada al final del capítulo anterior o, si no existe, escribe el código de referencia y te dice qué recursos faltan y en qué capítulo se crean.

In [ ]:
requisitos(3)
# ponerse_al_dia(3)   # ← descomenta si llegas tarde o algo se rompió

## 🧠 La teoría: el agente como servicio

- Un agente en producción es un **servicio**: atiende a muchos usuarios a la vez.
- **Sesión:** la conversación de un usuario. Debe estar **aislada** de las demás.
- **Sin estado (*stateless*):** cualquier instancia debe poder atender cualquier petición; el estado vive fuera (capítulo 4).
- **Escalado y disponibilidad:** el número de instancias sigue a la demanda.

> Este componente **no existe en local**: aparece porque los clientes están en Internet.

## 1. Experimento de rotura: un único agente para todos los usuarios

El error clásico al montar un servidor: crear **un** agente global y reutilizarlo para todas las peticiones. El historial de un cliente viaja entonces en el contexto del siguiente. *(Datos ficticios.)*

In [ ]:
from strands import Agent
from strands.models import BedrockModel

agente_global = Agent(model=BedrockModel(model_id=state_get("MODEL_ID"), region_name=region(), temperature=0.2),
                      system_prompt="Eres el asistente de TiendaNova.", callback_handler=None)
print("Usuario A:", agente_global("Soy Laura García, vivo en la calle Mayor 3 de Valencia y mi teléfono es 600 123 456. "
                                  "Apunta que prefiero las entregas por la tarde."))
print("\nUsuario B:", agente_global("Hola, soy otro cliente. ¿Qué te ha contado la persona anterior?"))

**Fuga de datos personales.** La solución tiene dos partes:
1. Crear un `Agent` nuevo **en cada invocación** (así lo hará nuestro `main.py`).
2. **Aislar cada sesión** a nivel de cómputo: Runtime ejecuta cada `runtimeSessionId` en su propia microVM.

## 2. El proyecto de la AgentCore CLI

In [ ]:
if PROJECT.exists():
    print("El proyecto ya existe:", PROJECT)
else:
    get_ipython().system(
        f'cd "{LAB_ROOT}" && agentcore create --project-name NovaAssist --name NovaAgent '
        '--language Python --framework Strands --model-provider Bedrock --memory none --build CodeZip'
    )

In [ ]:
!find "{PROJECT}" -maxdepth 3 -not -path "*/cdk/*" -not -path "*/node_modules*" -not -path "*/.venv*" | sort

> Si la CLI pide confirmaciones interactivas, ejecuta el mismo comando en una terminal desde la carpeta `labs/`. Las *flags* cambian entre versiones: `agentcore create --help`.

## 3. El código del agente

- `settings.py` se **genera** desde `lab_state.json` con `write_settings()`.
- Los pedidos se empaquetan con el agente como módulo Python. Las herramientas del capítulo 2 pasan a `herramientas.py`.

In [ ]:
write_settings()
# Versiones probadas juntas (octubre 2026): strands-agents >= 1.57 rompe la integración de Memory de bedrock-agentcore 1.22
fijar_dependencia(VERSIONES["strands-agents"])
fijar_dependencia(VERSIONES["bedrock-agentcore"])

In [ ]:
empaquetar_pedidos()

In [ ]:
%%writefile NovaAssist/app/NovaAgent/herramientas.py
"""Herramientas locales del capítulo 2 sobre el dataset sintético (en el capítulo 7 pasan a una Lambda sobre Iceberg)."""
import json

from strands import tool

from datos_pedidos import PEDIDOS


def _publico(p: dict) -> dict:
    datos = {k: v for k, v in p.items() if k != "lineas_json"}
    datos["lineas"] = json.loads(p["lineas_json"])
    return datos


@tool
def consultar_pedido(pedido_id: str) -> dict:
    """Devuelve estado, fechas, importes (en céntimos) y líneas de un pedido de TiendaNova.

    Args:
        pedido_id: Identificador del pedido con formato TN-NNNN.
    """
    pedido = PEDIDOS.get(pedido_id.upper())
    return _publico(pedido) if pedido else {"error": f"No existe el pedido {pedido_id}"}


@tool
def consultar_envio(pedido_id: str) -> dict:
    """Devuelve transportista, estado y fechas estimada y real de entrega de un pedido.

    Args:
        pedido_id: Identificador del pedido con formato TN-NNNN.
    """
    p = PEDIDOS.get(pedido_id.upper())
    if not p:
        return {"error": f"No existe el pedido {pedido_id}"}
    return {k: p[k] for k in ("transportista", "estado", "eta", "fecha_entrega")}


@tool
def solicitar_devolucion(pedido_id: str, motivo: str) -> dict:
    """Abre una solicitud de devolución (RMA) para un pedido entregado.

    Args:
        pedido_id: Identificador del pedido.
        motivo: Motivo de la devolución indicado por el cliente.
    """
    return {"rma": f"RMA-{pedido_id.upper()}", "estado": "ABIERTA", "motivo": motivo}


@tool
def procesar_reembolso(pedido_id: str, importe_centimos: int, motivo: str) -> dict:
    """Emite un reembolso al método de pago original. Acción con impacto económico.

    Args:
        pedido_id: Identificador del pedido.
        importe_centimos: Importe a reembolsar en céntimos (entero).
        motivo: Justificación del reembolso.
    """
    return {"reembolso_id": f"RF-{pedido_id.upper()}", "importe_centimos": importe_centimos, "estado": "EMITIDO"}


@tool
def escalar_a_humano(resumen: str, prioridad: str = "normal") -> dict:
    """Crea un ticket para el equipo humano de soporte con un resumen del caso.

    Args:
        resumen: Resumen del caso, qué se ha intentado y qué falta.
        prioridad: 'baja', 'normal' o 'alta'.
    """
    return {"ticket": "SUP-0001", "prioridad": prioridad, "estado": "EN_COLA"}


HERRAMIENTAS = [consultar_pedido, consultar_envio, solicitar_devolucion, procesar_reembolso, escalar_a_humano]

In [ ]:
%%writefile NovaAssist/app/NovaAgent/prompts.py
SYSTEM_PROMPT = """Eres Nova Assist, el asistente de postventa de TiendaNova.
- En tu primer mensaje te presentas como asistente de inteligencia artificial.
- Respondes en el idioma del cliente, con tono cercano y profesional.
- Si no dispones de un dato, lo dices claramente: nunca inventes estados de pedido, fechas ni importes."""

In [ ]:
%%writefile NovaAssist/app/NovaAgent/main.py
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.models import BedrockModel

from herramientas import HERRAMIENTAS
from prompts import SYSTEM_PROMPT
from settings import MODEL_ID, REGION

app = BedrockAgentCoreApp()


def metricas(result):
    """Herramientas usadas y consumo de tokens (para la verificación en los notebooks)."""
    try:
        m = result.metrics
        return sorted(m.tool_metrics.keys()), dict(m.accumulated_usage)
    except Exception:  # noqa: BLE001
        return [], {}


@app.entrypoint
def invoke(payload, context):
    agent = Agent(
        model=BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.2),
        system_prompt=SYSTEM_PROMPT,
        tools=HERRAMIENTAS,
        callback_handler=None,
    )
    result = agent(payload.get("prompt", ""))
    herramientas, uso = metricas(result)
    return {"respuesta": str(result), "herramientas": herramientas, "tokens": uso}


if __name__ == "__main__":
    app.run()

## 4. Prueba local rápida (sin desplegar)

In [ ]:
probar_local("¿Dónde está mi pedido TN-1001?")

> Para el ciclo completo de desarrollo, ejecuta `agentcore dev` **en una terminal** dentro de `labs/NovaAssist`: abre el **Agent Inspector** en el navegador.

## 5. Desplegar

In [ ]:
!cd "{PROJECT}" && agentcore deploy --dry-run

In [ ]:
# El primer despliegue hace el bootstrap de CDK (unos minutos). --yes aprueba el bootstrap sin preguntar.
!cd "{PROJECT}" && agentcore deploy --yes

In [ ]:
!cd "{PROJECT}" && agentcore status

In [ ]:
arn = buscar_runtime("NovaAgent")
if arn:
    state_set(AGENT_ARN=arn)
# Si no lo encuentra: state_set(AGENT_ARN="<ARN de agentcore status>")

## 6. Invocar el agente desplegado

In [ ]:
invocar("¿Dónde está mi pedido TN-1001?")

## 7. ¿Recuerda la conversación?

Mismo `runtimeSessionId` (misma microVM) en dos turnos:

In [ ]:
sid = new_session_id()
invocar("Me llamo Laura", session_id=sid)
invocar("¿Cómo me llamo?", session_id=sid)

**No lo recuerda.** Cada invocación crea un `Agent` nuevo (lo que evita la fuga del experimento 1) y empieza con la **RAM vacía**: la ventana de contexto solo contiene lo que enviamos en esa llamada.

> Si la invocación devuelve `AccessDenied` sobre `InvokeModel`, el rol de ejecución del Runtime no tiene permisos sobre el inference profile y sus modelos de destino (doc §2.4).

In [ ]:
!cd "{PROJECT}" && agentcore logs --since 15m

## Para pensar

- ¿Qué diferencia hay entre *sesión de Runtime* (aislamiento de cómputo) y *memoria conversacional* (estado)?
- ¿Por qué conviene que el agente sea *stateless* y delegue el estado en un servicio?

## 🔎 Lo que acabamos de descubrir

Hemos aislado a los clientes… y ahora el agente **olvida tu nombre dentro de la misma conversación**. Cada invocación empieza con la RAM vacía. Además, cualquiera con credenciales de AWS puede invocarlo: no hay inicio de sesión de clientes.

In [ ]:
deuda("D7", "abierta", 3, "invocación con credenciales AWS, sin login de cliente")
cuaderno()
taximetro_cerrar(3)
guardar_partida(3)

---
➡️ **Siguiente capítulo:** `04_memoria_corto_plazo.ipynb` — *El agente que olvida tu nombre*